In [4]:
#!/usr/bin/env python
# coding: utf-8

import string
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# =========================
# NLTK
# =========================
nltk.download('punkt')
nltk.download('stopwords')

# =========================
# DATASET LOAD
# =========================
df = pd.read_csv(
    'spamhamdata.csv',
    sep=';',
    engine='python',
    encoding='utf-8',
    on_bad_lines='skip'
)

if len(df.columns) >= 2:
    df.rename(columns={df.columns[0]: 'label', df.columns[1]: 'text'}, inplace=True)
elif len(df.columns) == 1:
    split_df = df.iloc[:, 0].astype(str).str.split('\t', n=1, expand=True)
    df = split_df
    df.columns = ['label', 'text']

df['text'] = df['text'].astype(str)
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

# =========================
# PREPROCESS
# =========================
stemmer = PorterStemmer()
stopwords_set = set(stopwords.words('english'))

corpus = []
for text in df['text']:
    tokens = text.lower().translate(
        str.maketrans('', '', string.punctuation)
    ).split()
    tokens = [stemmer.stem(w) for w in tokens if w not in stopwords_set]
    corpus.append(' '.join(tokens))

# =========================
# VECTORIZE & TRAIN
# =========================
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(corpus).toarray()
y = df['label_num'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

clf = RandomForestClassifier(n_estimators=200, n_jobs=-1)
clf.fit(X_train, y_train)

accuracy = clf.score(X_test, y_test)
print("Model accuracy:", accuracy)

# =========================
# GUI
# =========================
import tkinter as tk
from tkinter import messagebox

# ---------- Theme ----------
BG_COLOR = "#f5f7fa"
CARD_COLOR = "#ffffff"
PRIMARY = "#4f46e5"
SUCCESS = "#16a34a"
DANGER = "#dc2626"
TEXT_COLOR = "#111827"
MUTED = "#6b7280"

FONT_TITLE = ("Segoe UI", 14, "bold")
FONT_TEXT = ("Segoe UI", 11)
FONT_BUTTON = ("Segoe UI", 11, "bold")

history = []
history_visible = False

# ---------- Logic ----------
def preprocess_text(text):
    tokens = text.lower().translate(
        str.maketrans('', '', string.punctuation)
    ).split()
    return ' '.join(stemmer.stem(w) for w in tokens if w not in stopwords_set)

def classify_email():
    user_text = text_input.get("1.0", "end").strip()
    if not user_text:
        messagebox.showwarning("Warning", "Please enter an email text.")
        return

    processed = preprocess_text(user_text)
    x_email = vectorizer.transform([processed])

    proba = clf.predict_proba(x_email)[0]
    spam_prob = proba[1] * 100
    prediction = np.argmax(proba)

    result = "SPAM" if prediction == 1 else "NOT SPAM"
    color = DANGER if result == "SPAM" else SUCCESS

    result_label.config(text=result, bg=color)
    probability_label.config(text=f"Spam Probability: {spam_prob:.2f}%")

    history.append((user_text, result, spam_prob))
    history_listbox.insert(
        tk.END, f"{result} ({spam_prob:.1f}%) • {user_text[:30]}..."
    )

def clear_text():
    text_input.delete("1.0", "end")
    result_label.config(text="", bg=CARD_COLOR)
    probability_label.config(text="")

def clear_history():
    history.clear()
    history_listbox.delete(0, tk.END)
    clear_text()

def toggle_history():
    global history_visible
    if history_visible:
        history_frame.pack_forget()
        history_button.config(text="📂 Show History")
        history_visible = False
    else:
        history_frame.pack(side="right", padx=15, pady=15, fill="y")
        history_button.config(text="❌ Hide History")
        history_visible = True

def show_history(event):
    if not history_listbox.curselection():
        return
    idx = history_listbox.curselection()[0]
    mail, result, prob = history[idx]

    text_input.delete("1.0", "end")
    text_input.insert("1.0", mail)

    color = DANGER if result == "SPAM" else SUCCESS
    result_label.config(text=result, bg=color)
    probability_label.config(text=f"Spam Probability: {prob:.2f}%")

def enter_predict(event):
    classify_email()
    return "break"

# ---------- Window ----------
window = tk.Tk()
window.title("Spam Email Detector")
window.geometry("860x480")
window.configure(bg=BG_COLOR)

# ---------- Main Card ----------
card = tk.Frame(window, bg=CARD_COLOR)
card.pack(side="left", padx=20, pady=20)

title = tk.Label(
    card, text="📧 Spam Email Detector",
    font=FONT_TITLE, bg=CARD_COLOR, fg=TEXT_COLOR
)
title.pack(pady=(10, 5))

accuracy_label = tk.Label(
    card,
    text=f"Model Accuracy: {accuracy*100:.2f}%",
    font=("Segoe UI", 10, "bold"),
    bg=CARD_COLOR,
    fg=MUTED
)
accuracy_label.pack(pady=(0, 10))

text_input = tk.Text(
    card, height=9, width=55,
    font=FONT_TEXT, bd=0, bg="#f9fafb"
)
text_input.pack(padx=15)
text_input.bind("<Return>", enter_predict)

btn_frame = tk.Frame(card, bg=CARD_COLOR)
btn_frame.pack(pady=10)

predict_button = tk.Button(
    btn_frame, text="Predict",
    font=FONT_BUTTON, bg=PRIMARY, fg="white",
    bd=0, padx=20, pady=6,
    command=classify_email
)
predict_button.pack(side="left", padx=5)

clear_button = tk.Button(
    btn_frame, text="New Email",
    font=FONT_BUTTON, bg="#e5e7eb", fg=TEXT_COLOR,
    bd=0, padx=15, pady=6,
    command=clear_text
)
clear_button.pack(side="left", padx=5)

history_button = tk.Button(
    btn_frame, text="📂 Show History",
    font=FONT_BUTTON, bg="#e5e7eb", fg=TEXT_COLOR,
    bd=0, padx=15, pady=6,
    command=toggle_history
)
history_button.pack(side="left", padx=5)

result_label = tk.Label(
    card, text="", width=20,
    font=("Segoe UI", 13, "bold"),
    bg=CARD_COLOR, fg="white", pady=8
)
result_label.pack(pady=(5, 5))

probability_label = tk.Label(
    card, text="",
    font=("Segoe UI", 11),
    bg=CARD_COLOR,
    fg=TEXT_COLOR
)
probability_label.pack(pady=(0, 15))

# ---------- History ----------
history_frame = tk.Frame(window, bg=CARD_COLOR)

history_title = tk.Label(
    history_frame, text="🕘 History",
    font=FONT_TITLE, bg=CARD_COLOR
)
history_title.pack(pady=(10, 5))

history_listbox = tk.Listbox(
    history_frame, width=40, height=16,
    font=("Segoe UI", 10),
    bd=0, bg="#f9fafb"
)
history_listbox.pack(padx=10)
history_listbox.bind("<<ListboxSelect>>", show_history)

clear_history_button = tk.Button(
    history_frame, text="🧹 Clear History",
    font=FONT_BUTTON,
    bg="#fee2e2", fg=DANGER,
    bd=0, padx=15, pady=6,
    command=clear_history
)
clear_history_button.pack(pady=(6, 10))

window.mainloop()


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\90545\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\90545\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Model accuracy: 0.9741873804971319
